# Lab 01 — Campus Learning Resource Finder
**Boolean retrieval with Arabic and English records**

<div dir="rtl" lang="ar">

مختبر ١ — البحث في موارد التعلم الجامعية<br>سنساعد طلبة الكلية على اختيار موارد مناسبة لحاجاتهم باستخدام مصفوفة الوقوع والعمليات المنطقية. التعليمات بالعربية والإنجليزية، والشيفرة والمصطلحات التقنية بالإنجليزية.

</div>

## What you will do
Represent 12 resource records, implement four functions, run four search tasks, and explain matching versus relevance. Suggested time: 60–75 minutes, plus an optional five-minute warm-up.

<div dir="rtl" lang="ar">

الأهداف: تمثيل ١٢ موردا تعليميا، وإكمال أربع دوال، وتنفيذ أربع مهام بحث، ثم تفسير الفرق بين المطابقة والصلة. هذه امتداد عملي للمحاضرة الأولى. الفهرس المعكوس يأتي في المحاضرة الثانية.

</div>

## 0. Optional bridge to the slides
Open `data/lecture_warmup.json`. On paper, write the wink/drink/ink rows and evaluate `wink AND drink AND NOT ink`. Then move to the realistic collection below.

<div dir="rtl" lang="ar">

تمهيد اختياري: استخدم مجموعة الوثائق الخمس في المحاضرة لكتابة ثلاثة صفوف ثنائية وحل الاستعلام على الورق. بعد ذلك انتقل إلى مجموعة الموارد التعليمية. لا يلزم إنشاء مجموعة أخرى من الدوال لهذا التمهيد.

</div>

## 1. Inspect the catalog
Open `START_HERE_AR_EN.html` in a browser for the resource table and task descriptions. Each JSON record has an ID, title, summary, language, level, access, format, topics and estimated study time. The records are fictional teaching data, not links to real courses.

<div dir="rtl" lang="ar">

ابدأ بفتح دليل المختبر في المتصفح لقراءة جدول الموارد. اقرأ العنوان والوصف، ثم لاحظ اللغة والمستوى ونوع المورد وإتاحته ووسوم الموضوع. البيانات افتراضية للتدريب وتحاكي فهرس موارد في بوابة تعليمية.

</div>

### Supplied helpers
Run this cell unchanged. The tokenizer supports Unicode letters. It lowercases English and removes punctuation. It does not translate, stem, normalize Arabic spelling variants or remove stop words.

<div dir="rtl" lang="ar">

دوال مساعدة جاهزة: شغل الخلية كما هي. تدعم معالجة الحروف العربية والإنجليزية، لكنها لا تترجم ولا توحد الصيغ العربية المختلفة. وسوم الموضوع محررة يدويا في الفهرس، وليست نتيجة ترجمة آلية أو نموذج دلالي.

</div>

In [ ]:
import json, re
from pathlib import Path
# er = Regular Expressions 

resources = json.loads(Path("data/learning_resources.json").read_text(encoding="utf-8"))
search_tasks = json.loads(Path("data/search_tasks.json").read_text(encoding="utf-8"))
doc_ids = [record["id"] for record in resources]   # List Comprehension to extract the "id" field from each record in resources

def tokenize(text):
    # Unicode letters support both Arabic and English; no stemming or translation.
    return re.findall(r"[^\W\d_]+", text.casefold(), flags=re.UNICODE)

def features(record, include_tags=True):
    result = set(tokenize(record["title"] + " " + record["summary"]))
    if include_tags:
        # Controlled metadata labels supplied by the catalog editor.
        result.update("topic_" + topic for topic in record["topics"])
        result.update({"lang_" + record["language"], "level_" + record["level"],
                       "access_" + record["access"], "type_" + record["format"]})
    return result

def matching_ids(vector):
    if len(vector) != len(doc_ids):
        raise ValueError("A result vector needs one value per resource")
    return [doc_id for doc_id, flag in zip(doc_ids, vector) if flag]

def show_results(vector):
    selected = set(matching_ids(vector))
    for record in resources:
        if record["id"] in selected:
            print(record["id"], record["title"])

def print_matrix(matrix):
    print(f"{'Term or tag':<19}", *(f"{doc_id:>3}" for doc_id in doc_ids))
    for term, vector in matrix.items():
        print(f"{term:<19}", *(f"{value:>3}" for value in vector))

print("Resource IDs:", doc_ids)
print("Arabic example:", tokenize("تعلم بايثون"))
print("English example:", tokenize("Learn Python"))


Resource IDs: ['R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08', 'R09', 'R10', 'R11', 'R12']
Arabic example: ['تعلم', 'بايثون']
English example: ['learn', 'python']


In [5]:
print(features(resources[0]))
print(features(resources[0], include_tags=False))
features(resources[1], include_tags=False)

{'قصيرة', 'بايثون', 'والحلقات', 'level_beginner', 'إلى', 'مع', 'فيديو', 'مدخل', 'أمثلة', 'access_free', 'topic_python', 'lang_ar', 'للمبتدئين', 'والشروط', 'type_video', 'المتغيرات', 'يشرح'}
{'قصيرة', 'بايثون', 'والحلقات', 'إلى', 'مع', 'فيديو', 'مدخل', 'أمثلة', 'للمبتدئين', 'والشروط', 'المتغيرات', 'يشرح'}


{'a',
 'and',
 'basics',
 'by',
 'conditions',
 'exercises',
 'followed',
 'for',
 'loops',
 'on',
 'python',
 'short',
 'students',
 'tutorial',
 'university',
 'variables',
 'written'}

## 2. Exercise A — Incidence matrix
Complete `incidence_matrix`. Rows are vocabulary terms or tags. Columns are resources in their original order. Use 1 for presence and 0 for absence.

<div dir="rtl" lang="ar">

أكمل دالة المصفوفة. كل صف يمثل كلمة أو وسما، وكل عمود يمثل موردا. تكرار الكلمة لا يزيد القيمة عن واحد. لا تغير ترتيب الموارد أثناء العمليات.

</div>

In [ ]:
def incidence_matrix(records, vocabulary, include_tags=True):
    # TODO: Use features(record, include_tags) for each resource.
    feature_sets = [ features(record, include_tags)  for record in records]
    matrix = {}
    
    for term in vocabulary:
            matrix[term] = [
                int(term in feature_set)
                for feature_set in feature_sets]
    return matrix

In [9]:
vocabulary = [
    "بايثون", "python", "topic_python", "topic_ir", "topic_nlp", "topic_web",
    "lang_ar", "lang_en", "level_beginner", "level_advanced", "access_free",
    "access_paid", "type_notebook"
]
matrix = incidence_matrix(resources, vocabulary)
print_matrix(matrix)


Term or tag         R01 R02 R03 R04 R05 R06 R07 R08 R09 R10 R11 R12
بايثون                1   0   1   0   0   0   0   0   1   0   1   1
python                0   1   0   0   0   0   0   0   0   1   0   0
topic_python          1   1   1   0   0   0   0   0   1   1   1   0
topic_ir              0   0   0   1   1   1   0   0   0   0   0   0
topic_nlp             0   0   0   0   0   1   0   0   0   0   1   0
topic_web             0   0   0   0   0   0   1   1   0   0   0   0
lang_ar               1   0   1   0   1   0   1   0   1   0   1   1
lang_en               0   1   0   1   0   1   0   1   0   1   0   0
level_beginner        1   1   0   1   1   0   0   1   1   0   0   1
level_advanced        0   0   0   0   0   1   0   0   0   0   0   0
access_free           1   1   1   1   1   1   0   1   1   0   1   1
access_paid           0   0   0   0   0   0   1   0   0   1   0   0
type_notebook         0   0   1   0   0   1   0   0   1   0   1   0


### How metadata enters the representation
`topic_python` is a catalog topic tag, `lang_ar` means Arabic, `level_beginner` means beginner, `access_free` means free, and `type_notebook` means a notebook. These labels are deliberately supplied so Lecture 1 can combine text and simple attributes.

<div dir="rtl" lang="ar">

تضاف السمات المنظمة إلى التمثيل على شكل وسوم واضحة. نستخدم هذه الوسوم كي نترجم حاجة الطالب إلى شروط قابلة للفحص. مطابقة وسم موضوع عبر لغتين لا تعني أن النظام يفهم اللغتين أو يترجم بينهما.

</div>

In [11]:
print("Arabic word:", matching_ids(matrix["بايثون"]))
print("English word:", matching_ids(matrix["python"]))
print("Topic tag:", matching_ids(matrix["topic_python"]))

Arabic word: ['R01', 'R03', 'R09', 'R11', 'R12']
English word: ['R02', 'R10']
Topic tag: ['R01', 'R02', 'R03', 'R09', 'R10', 'R11']


## 3. Exercise B — Boolean operations
Complete AND, OR and NOT. Reject unequal input lengths for AND and OR. NOT complements the current collection, not every possible document.

<div dir="rtl" lang="ar">

أكمل التقاطع والاتحاد والنفي. يجب أن يكون للمتجهين الطول نفسه. النفي هنا يعني الموارد غير المطابقة داخل المجموعة الحالية فقط.

</div>

In [12]:
def and_vector(a, b):
    # TODO: Reject unequal lengths and apply AND column by column.
    if len(a) != len(b):
        raise ValueError("Vectors must have equal length")

    return [int(x and y) for x, y in zip(a, b)]
    # raise NotImplementedError("Complete AND")

def or_vector(a, b):
    # TODO: Reject unequal lengths and apply OR column by column.
    if len(a) != len(b):
        raise ValueError("Vectors must have equal length")

    return [int(x or y) for x, y in zip(a, b)]

def not_vector(a):
    # TODO: Complement each value within the current resource collection.
    return [1 - x for x in a]
    


In [13]:
assert and_vector([1, 1, 0], [1, 0, 1]) == [1, 0, 0]
assert or_vector([1, 1, 0], [1, 0, 1]) == [1, 1, 1]
assert not_vector([1, 0, 0]) == [0, 1, 1]
assert len(matrix["topic_python"]) == len(resources)
assert set(matrix["topic_python"]) <= {0, 1}
assert incidence_matrix(resources, ["missing_term"])["missing_term"] == [0] * len(resources)
for operation in (and_vector, or_vector):
    try:
        operation([1], [1, 0])
        raise AssertionError("Reject unequal vector lengths")
    except ValueError:
        pass
print("Basic checks passed. Review the matrix against the catalog too.")


Basic checks passed. Review the matrix against the catalog too.


## 4. Worked task Q1
A beginner needs a free Arabic resource for learning Python. Predict the returned IDs, then run the formula. Explain why each matching record satisfies all four conditions.

<div dir="rtl" lang="ar">

مهمة محلولة: طالب مبتدئ يحتاج موردا مجانيا باللغة العربية لتعلم بايثون. توقع المعرفات قبل التشغيل، ثم فسر كل شرط في الاستعلام. قارِن بين ملاءمة فيديو تمهيدي ودفتر يتطلب كتابة شيفرة.

</div>

In [14]:
# Q1 is a worked example. Predict the IDs before running it.
q1 = and_vector(
    and_vector(matrix["topic_python"], matrix["level_beginner"]),
    and_vector(matrix["lang_ar"], matrix["access_free"])
)
show_results(q1)


R01 مدخل إلى بايثون
R09 بايثون للمبتدئين تدريبات عملية


## 5. Exercise C — Tasks Q2, Q3 and Q4
Read `data/search_tasks.json`. Translate the three expressions into calls to your functions, predict the IDs and inspect the results. Parentheses matter. No parser, inverted index, ranking model or external search library is required.

<div dir="rtl" lang="ar">

أكمل مهام البحث الثلاث في ملف المهام. استخدم الدوال التي كتبتها وراع ترتيب الأقواس. سجل توقعك أولا، ثم النتيجة الفعلية. المطلوب فهم العمليات المنطقية، ولا نحتاج في هذا المختبر إلى محلل استعلامات أو فهرس معكوس أو نموذج ترتيب.

</div>

In [15]:
# TODO: Translate Q2, Q3 and Q4 into calls to your Boolean functions.
# Read the parentheses carefully. Do not use eval() or a query parser.
q2 = and_vector(
    and_vector(matrix["topic_ir"], or_vector(matrix["lang_ar"], matrix["lang_en"])),
    and_vector(not_vector( matrix["level_advanced"]), matrix["access_free"])
)
# for q2 ==> expected : r4 , r5 
# --------------------------------
q3 = and_vector( # (topic_python OR topic_nlp) AND type_notebook AND access_free
   or_vector(matrix["topic_python"], matrix["topic_nlp"]),
   and_vector(matrix["type_notebook"], matrix["access_free"])
)     # expected for q3 ==> r3,r6, r9, r11
# ------------------------------
q4 = and_vector( # topic_web AND access_free AND NOT access_paid
matrix["topic_web"], 
and_vector(matrix["access_free"], not_vector(matrix["access_paid"]))
)
# for q4 ==> expected : r8

# -------------------------------------------------
if any(vector is None for vector in (q2, q3, q4)):
    raise NotImplementedError("Complete Q2, Q3 and Q4, then rerun this cell")
for query_id, vector in [("Q2", q2), ("Q3", q3), ("Q4", q4)]:
    print(query_id)
    show_results(vector)


Q2
R04 Introduction to Information Retrieval
R05 مقدمة في استرجاع المعلومات
Q3
R03 تحليل البيانات باستخدام بايثون
R06 Neural Ranking with Transformers
R09 بايثون للمبتدئين تدريبات عملية
R11 معالجة اللغة العربية باستخدام بايثون
Q4
R08 Web Security Essentials


In [17]:
my_query = and_vector(
    and_vector(matrix["topic_ir"], matrix["lang_en"]),
    matrix["access_free"]
)

show_results(my_query)

R04 Introduction to Information Retrieval
R06 Neural Ranking with Transformers


## 6. Experiment — Vocabulary mismatch
Run the next cell and compare the English word, Arabic word, either word and curated topic tag. Explain the extra word match and the resources missed by a single-language query.

<div dir="rtl" lang="ar">

جرب البحث بكلمة إنجليزية ثم عربية، ثم اتحادهما، ثم وسم الموضوع. هل ظهرت الموارد نفسها؟ اقرأ وصف المورد R12: هل ذكر اسم اللغة يجعله مادة مناسبة لتعلمها؟ اربط ذلك بالحاجة المعلوماتية وفجوة المفردات.

</div>

In [16]:
# Here the first two rows use only title/summary words; the third uses a catalog tag.
text_rows = incidence_matrix(resources, ["python", "بايثون"], include_tags=False)
english_hits = text_rows["python"]
arabic_hits = text_rows["بايثون"]
combined_hits = or_vector(english_hits, arabic_hits)
tag_hits = matrix["topic_python"]
for label, vector in [("English word", english_hits), ("Arabic word", arabic_hits),
                      ("Either word", combined_hits), ("Curated topic tag", tag_hits)]:
    print(label, matching_ids(vector))


English word ['R02', 'R10']
Arabic word ['R01', 'R03', 'R09', 'R11', 'R12']
Either word ['R01', 'R02', 'R03', 'R09', 'R10', 'R11', 'R12']
Curated topic tag ['R01', 'R02', 'R03', 'R09', 'R10', 'R11']


## 7. Explain and submit
Complete `Worksheet_AR_EN.md`. Submit the notebook with your code and outputs plus the worksheet. Optional extension: add one fictional resource, rerun, and explain how the universe of NOT changed.

<div dir="rtl" lang="ar">

أجب عن ورقة العمل وأرفق دفتر الشيفرة بعد حفظ المخرجات. اشرح اختيارك للمورد الأفضل لحاجة محددة، وحدود هذا النظام. للتوسع أضف موردا افتراضيا واحدا، ثم وضح أثره في النتائج وفي مجموعة النفي.

</div>